In [1]:
import xarray as xr
import pandas as pd
import dask

In [2]:
latitude = [-22, -23, -24, -25]
longitude = [142, 143, 144, 145]


In [3]:
pr_path = "/g/data/ux62/access-s2/hindcast/raw_model/atmos/pr/monthly"
output_path = "output_data/"

ensemble_count = 1
year_range = (1993, 1993)

for e_i in range(1, ensemble_count + 1):
    for y in range(year_range[0], year_range[1] + 1):
        month_start_dates = xr.date_range(f"{y}-01-01", end=f"{y}-12-31", freq="MS")
        for m_dt in month_start_dates:
            m = m_dt.month
            e = f"e{e_i:02d}"
            dataset_paths = []
            # Last 8 days of the preceding month
            for day in range(m_dt.days_in_month-7, m_dt.days_in_month + 1):
                ds = xr.open_dataset(f"{pr_path}/{e}/ma_pr_{y}{m:02d}{day}_{e}.nc")
                # Spatial
                ds = ds.sel(lat=latitude, lon=longitude, method="nearest")
                # Timings
                ds = ds.isel(time=slice(0, 6))
                print(ds)
            n_m = (m % 12) + 1
            n_y = y + 1 if m == 12 else y
            ds = xr.open_dataset(f"{pr_path}/{e}/ma_pr_{n_y}{n_m:02d}01_{e}.nc")
            # Select nearest points for 1 degree resolution and 6-time
            # TODO: Remove copy from above
            ds = ds.sel(lat=latitude, lon=longitude, method="nearest")
            ds = ds.isel(time=slice(0, 6))

/home/659/ag9761/.local/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


<xarray.Dataset> Size: 464B
Dimensions:  (time: 6, lat: 4, lon: 4)
Coordinates:
  * lat      (lat) float32 16B -21.94 -23.06 -24.17 -24.72
  * lon      (lon) float32 16B 142.1 142.9 143.8 145.4
  * time     (time) datetime64[ns] 48B 1993-02-15 ... 1993-07-16T12:00:00
Data variables:
    pr       (time, lat, lon) float32 384B ...
Attributes: (12/26)
    model_id:                ACCESS-S2
    distribution_statement:  The data are subject to the distribution terms o...
    creator_name:            Bureau of Meteorology
    copyright:               (C) Copyright Commonwealth of Australia (2017), ...
    acknowledgement:         Australian Bureau of Meteorology, Commonwealth o...
    title:                   ACCESS-S2
    ...                      ...
    input_uri:               file:///g/sc/home_trial/coupled_trial/cylc-run/s...
    input_word_length:       8
    history:                 Fri Apr 30 08:19:42 2021: ncks -O -v pr pc199301...
    input_byte_ordering:     big_endian
    NCO:   

In [14]:
pr_path = "/g/data/ux62/access-s2/hindcast/raw_model/atmos/pr/monthly"
output_path = "output_data/"

ensemble_count = 3
year_range = (1993, 1993)


def get_dataset_paths(m_dt, e_i):
    m = m_dt.month
    e = f"e{e_i:02d}"
    dataset_paths = []
    # Last 8 days of the preceding month
    for day in range(m_dt.days_in_month-7, m_dt.days_in_month + 1):
        dataset_paths.append(f"{pr_path}/{e}/ma_pr_{y}{m:02d}{day}_{e}.nc")
    n_m = (m % 12) + 1
    n_y = y + 1 if m == 12 else y
    # First day of the next month
    dataset_paths.append(f"{pr_path}/{e}/ma_pr_{n_y}{n_m:02d}01_{e}.nc")
    return dataset_paths


# Within each month, collect all ensembles
yearly_dataset = []
for y in range(year_range[0], year_range[1] + 1):
    month_start_dates = xr.date_range(f"{y}-01-01", end=f"{y}-12-31", freq="MS")
    for m_dt in month_start_dates:
        preprocessed_ensemble_list = []
        for e_i in range(1, ensemble_count + 1):
            dataset_paths = get_dataset_paths(m_dt, e_i)
            datasets = [xr.open_dataset(f) for f in dataset_paths]
            ds = xr.concat(datasets, dim="ensemble_id")

            # Nearest grid point selection (assumption of top-left)
            ds = ds.sel(lat=latitude, lon=longitude, method="nearest")

            # First 6 months forecast time
            ds = ds.isel(time=slice(0, 6))
            preprocessed_ensemble_list.append(ds)
        ds = xr.concat(preprocessed_ensemble_list, dim="ensemble_id")
        print(ds)
        yearly_dataset.append(ds)

ds = xr.concat(yearly_dataset, dim="indexing_time")
ds

<xarray.Dataset>
Dimensions:  (lat: 4, lon: 4, time: 6, ensemble_id: 27)
Coordinates:
  * lat      (lat) float32 -21.94 -23.06 -24.17 -24.72
  * lon      (lon) float32 142.1 142.9 143.8 145.4
  * time     (time) datetime64[ns] 1993-02-15 ... 1993-07-16T12:00:00
Dimensions without coordinates: ensemble_id
Data variables:
    pr       (ensemble_id, time, lat, lon) float32 2.765e-05 ... 7.313e-08
Attributes: (12/26)
    model_id:                ACCESS-S2
    distribution_statement:  The data are subject to the distribution terms o...
    creator_name:            Bureau of Meteorology
    copyright:               (C) Copyright Commonwealth of Australia (2017), ...
    acknowledgement:         Australian Bureau of Meteorology, Commonwealth o...
    title:                   ACCESS-S2
    ...                      ...
    input_uri:               file:///g/sc/home_trial/coupled_trial/cylc-run/s...
    input_word_length:       8
    history:                 Fri Apr 30 08:19:42 2021: ncks -O -v 

<xarray.Dataset>
Dimensions:  (lat: 4, lon: 4, time: 17, indexing_time: 12, ensemble_id: 27)
Coordinates:
  * lat      (lat) float32 -21.94 -23.06 -24.17 -24.72
  * lon      (lon) float32 142.1 142.9 143.8 145.4
  * time     (time) datetime64[ns] 1993-02-15 1993-03-16T12:00:00 ... 1994-06-16
Dimensions without coordinates: indexing_time, ensemble_id
Data variables:
    pr       (indexing_time, ensemble_id, time, lat, lon) float32 2.765e-05 ....
Attributes: (12/26)
    model_id:                ACCESS-S2
    distribution_statement:  The data are subject to the distribution terms o...
    creator_name:            Bureau of Meteorology
    copyright:               (C) Copyright Commonwealth of Australia (2017), ...
    acknowledgement:         Australian Bureau of Meteorology, Commonwealth o...
    title:                   ACCESS-S2
    ...                      ...
    input_uri:               file:///g/sc/home_trial/coupled_trial/cylc-run/s...
    input_word_length:       8
    history:                 Fri Apr 30 08:19:42 2021: ncks -O -v pr pc199301...
    input_byte_ordering:     big_endian
    NCO:                     "4.5.3"
    monthly_average:         Monthly file built with script ./process_missing...

In [7]:
preprocessed_ensembles = ds = xr.concat(_preprocessed_ensembles, dim="ensemble_id")

In [8]:
preprocessed_ensembles

<xarray.Dataset>
Dimensions:  (lat: 4, lon: 4, time: 17, ensemble_id: 108)
Coordinates:
  * lat      (lat) float32 -21.94 -23.06 -24.17 -24.72
  * lon      (lon) float32 142.1 142.9 143.8 145.4
  * time     (time) datetime64[ns] 1993-02-15 1993-03-16T12:00:00 ... 1994-06-16
Dimensions without coordinates: ensemble_id
Data variables:
    pr       (ensemble_id, time, lat, lon) float32 2.765e-05 ... 1.095e-05
Attributes: (12/26)
    model_id:                ACCESS-S2
    distribution_statement:  The data are subject to the distribution terms o...
    creator_name:            Bureau of Meteorology
    copyright:               (C) Copyright Commonwealth of Australia (2017), ...
    acknowledgement:         Australian Bureau of Meteorology, Commonwealth o...
    title:                   ACCESS-S2
    ...                      ...
    input_uri:               file:///g/sc/home_trial/coupled_trial/cylc-run/s...
    input_word_length:       8
    history:                 Fri Apr 30 08:19:42 2021: ncks -O -v pr pc199301...
    input_byte_ordering:     big_endian
    NCO:                     "4.5.3"
    monthly_average:         Monthly file built with script ./process_missing...